# Synthetic populations against Prompt 264

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json, math
import numpy as np
from scipy.special import ndtr
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.observation" and doc.get("kind") == "table", "this notebook reads a gravitas.observation/1 table"
cols = {c["id"]: np.array(c["values"], float) for c in doc["columns"]}
x, sig = cols["x"], cols["sigma"]
des = doc["design"]
edges = np.array(des["edges"], float)
E = np.array(des["exposure"], float)
r = np.array(des["reliability"], float)
J = len(edges) - 1
assert len(x) >= 1, "refused: there are no objects"
assert bool(np.all(np.diff(edges) > 0)) and len(E) == J and len(r) == J, "refused: the design's edges, exposure and reliability do not agree"
assert bool(np.all(E > 0)), "refused: an exposure is not positive"
assert bool(np.all(r > 0)), "refused: a reliability is not positive"
assert bool(np.all(sig >= 0)), "refused: an uncertainty is negative"
width = np.diff(edges)

In [ ]:
def kernel(xv, sv):
    K = np.zeros((len(xv), J))
    for j in range(J):
        a, b = edges[j], edges[j + 1]
        pos = sv > 0
        K[pos, j] = (ndtr((b - xv[pos]) / sv[pos]) - ndtr((a - xv[pos]) / sv[pos])) / width[j]
        K[~pos, j] = ((xv[~pos] >= a) & (xv[~pos] < b)) / width[j]
    return K

def fit(xv, sv):
    K = kernel(xv, sv)
    mu = np.full(J, len(xv) / J)
    last = -np.inf
    for it in range(1, 20001):
        s = K @ mu
        ll = float(np.log(s).sum() - mu.sum())
        mu = mu * (K.T @ (1.0 / s))
        if abs(ll - last) <= 1e-12 * max(1.0, abs(ll)):
            break
        last = ll
    s = K @ mu
    return mu, float(np.log(s).sum() - mu.sum()), it

mu_hat, ll, iters = fit(x, sig)
status = ["not constrained" if E[j] < 5 else "ok" for j in range(J)]
gamma = [float(mu_hat[j] * r[j] / E[j]) if status[j] == "ok" else None for j in range(J)]
ans = {"mu": mu_hat.tolist(), "gamma": gamma, "status": status, "logLik": ll, "iterations": iters}
if "truth" in doc and len(doc["truth"]["muTrue"]) == J and des.get("exposure") == [100.0] * J:
    mu_true = np.array(doc["truth"]["muTrue"], float)
    Z = []
    for rep in range(200):
        rng = np.random.default_rng(268001 + rep)
        counts = rng.poisson(mu_true)
        us = np.concatenate([rng.uniform(edges[j], edges[j + 1], counts[j]) for j in range(J)])
        bw = np.repeat(width, counts)
        sg = bw * np.exp(rng.uniform(math.log(0.1), math.log(0.3), len(us)))
        xs = us + sg * rng.standard_normal(len(us))
        Z.append((fit(xs, sg)[0] - mu_true) / np.sqrt(mu_true))
    Z = np.array(Z)
    ans["recovery"] = {"replicates": 200, "meanZ": Z.mean(axis=0).tolist(), "rmsZ": np.sqrt((Z ** 2).mean(axis=0)).tolist(), "pooledRmsZ": float(np.sqrt((Z ** 2).mean()))}
gravitas_output = {"answer": ans}